# 🔗 MinIO S3 Data Explorer

**Mục đích**: Kết nối MinIO, tải parquet từ data lake và hiển thị nội dung

Notebook này cung cấp một môi trường interactive để:
- Kết nối với MinIO S3
- Liệt kê buckets và files
- Tải dữ liệu Parquet
- Hiển thị dữ liệu chi tiết

In [11]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

packages = ['minio', 'pandas', 'pyarrow', 'confluent_kafka', 'confluent-kafka']

print("📦 Cài đặt thư viện...\n")
for package in packages:
    try:
        __import__(package)
        print(f"✅ {package} - đã có sẵn")
    except ImportError:
        print(f"⬇️  Đang cài {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} - cài xong\n")

print("\n✨ Tất cả thư viện sẵn sàng!")

📦 Cài đặt thư viện...

✅ minio - đã có sẵn
✅ pandas - đã có sẵn
✅ pyarrow - đã có sẵn
✅ confluent_kafka - đã có sẵn
⬇️  Đang cài confluent-kafka...
✅ confluent-kafka - cài xong


✨ Tất cả thư viện sẵn sàng!


In [14]:
# Import thư viện
import os
from minio import Minio
from minio.error import S3Error
import pandas as pd
import io

# Cấu hình kết nối MinIO
MINIO_CONFIG = {
    'endpoint': os.getenv('MINIO_ENDPOINT', 'localhost:9000'),
    'access_key': os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    'secret_key': os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    'region': os.getenv('MINIO_REGION', 'ap-southeast-1'),
    'use_ssl': False
}

print(f"🔧 Kết nối MinIO tại: {MINIO_CONFIG['endpoint']}")

try:
    # Khởi tạo client MinIO
    client = Minio(
        endpoint=MINIO_CONFIG['endpoint'],
        access_key=MINIO_CONFIG['access_key'],
        secret_key=MINIO_CONFIG['secret_key'],
        region=MINIO_CONFIG['region'],
        secure=MINIO_CONFIG['use_ssl']
    )
    
    print("✅ Kết nối MinIO thành công!")
    
except Exception as err:
    print(f"❌ Lỗi kết nối: {err}")
    print(f"💡 Đảm bảo MinIO đang chạy tại {MINIO_CONFIG['endpoint']}")

🔧 Kết nối MinIO tại: localhost:9000
✅ Kết nối MinIO thành công!


In [15]:
# Liệt kê tất cả buckets
print("📦 Danh sách Buckets:\n")

try:
    buckets = list(client.list_buckets())
    
    if buckets:
        for i, bucket in enumerate(buckets, 1):
            print(f"{i}. {bucket.name}")
        print(f"\n✅ Tổng cộng: {len(buckets)} bucket")
    else:
        print("⚠️  Không có bucket nào. Tạo 'data-lake' bucket...")
        client.make_bucket('data-lake')
        print("✅ Bucket 'data-lake' đã tạo")
        
except Exception as err:
    print(f"❌ Lỗi: {err}")

📦 Danh sách Buckets:

1. data-lake

✅ Tổng cộng: 1 bucket


In [16]:
# Tìm và tải file parquet
print("🔍 Tìm file Parquet...\n")

bucket_name = 'data-lake'
parquet_files = []

try:
    # Tìm tất cả file .parquet
    for obj in client.list_objects(bucket_name, recursive=True):
        if obj.object_name.endswith('.parquet'):
            parquet_files.append(obj.object_name)
    
    if parquet_files:
        print(f"✅ Tìm thấy {len(parquet_files)} file Parquet:\n")
        for i, file in enumerate(parquet_files[:5], 1):
            print(f"{i}. {file}")
        
        if len(parquet_files) > 5:
            print(f"... và {len(parquet_files) - 5} file khác")
    else:
        print("⚠️  Chưa có file Parquet. Hãy chạy Spark job để tạo dữ liệu.")
        
except S3Error as err:
    print(f"❌ Lỗi: {err}")

🔍 Tìm file Parquet...

✅ Tìm thấy 40 file Parquet:

1. warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=delivery-status.v1/00099-1325-3795892e-88ff-4760-b297-1d0dd803866d-3-00001.parquet
2. warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=delivery-status.v1/00099-15-3795892e-88ff-4760-b297-1d0dd803866d-0-00001.parquet
3. warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=delivery-status.v1/00099-1761-3795892e-88ff-4760-b297-1d0dd803866d-4-00001.parquet
4. warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=delivery-status.v1/00099-2197-3795892e-88ff-4760-b297-1d0dd803866d-5-00001.parquet
5. warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=delivery-status.v1/00099-2633-3795892e-88ff-4760-b297-1d0dd803866d-6-00001.parquet
... và 35 file khác


In [18]:
# Hàm tải và hiển thị nội dung Parquet
def load_and_display_parquet(file_path: str, limit: int = 20):
    """
    Tải file Parquet từ MinIO và hiển thị nội dung
    
    Args:
        file_path: Đường dẫn file trong bucket
        limit: Số dòng hiển thị
    """
    try:
        print(f"⬇️  Tải: {file_path}")
        print("-" * 80)
        
        # Download file từ MinIO
        response = client.get_object(bucket_name, file_path)
        data = response.read()
        
        # Load vào DataFrame
        df = pd.read_parquet(io.BytesIO(data))
        
        print(f"\n📊 Thông tin:")
        print(f"   Số dòng: {len(df):,}")
        print(f"   Số cột: {df.shape[1]}")
        print(f"   Dung lượng: {len(data) / 1024 / 1024:.2f} MB")
        
        print(f"\n📋 Cấu trúc dữ liệu:")
        print(df.dtypes)
        
        print(f"\n📄 Dữ liệu (hiển thị {limit} dòng đầu):")
        print(df.head(limit).to_string())
        
        return df
        
    except Exception as err:
        print(f"❌ Lỗi: {err}")
        return None

# Tải file Parquet đầu tiên nếu có
if parquet_files:
    df = load_and_display_parquet(parquet_files[30])
else:
    print("⚠️  Chưa có file Parquet để tải")

⬇️  Tải: warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=payments.v1/00172-2635-3795892e-88ff-4760-b297-1d0dd803866d-6-00001.parquet
--------------------------------------------------------------------------------

📊 Thông tin:
   Số dòng: 1,082
   Số cột: 7
   Dung lượng: 0.03 MB

📋 Cấu trúc dữ liệu:
event_source                    str
event_time      datetime64[us, UTC]
schema_id                     int32
payload_size                  int32
json_payload                    str
partition                     int32
offset                        int64
dtype: object

📄 Dữ liệu (hiển thị 20 dòng đầu):
   event_source                       event_time  schema_id  payload_size                                                                                                                     json_payload  partition  offset
0   payments.v1 2026-04-03 16:42:12.127000+00:00          0           127  {"payment_id": "pay_m0mc6bnpcw", "order_id": "ord_mrnwm6xyaf", "method": "A

In [6]:
# Hàm in dữ liệu theo từng topic
TOPICS = {
    'orders': 'orders',
    'payments': 'payments',
    'shipments': 'shipments',
    'delivery': 'delivery',
    'users': 'users',
    'products': 'products'
}

def list_all_parquet_files():
    """
    Hiển thị tất cả file Parquet tìm được
    """
    print(f"\n🔍 TẤT CẢ FILE PARQUET ({len(parquet_files)} files):\n")
    
    if not parquet_files:
        print("⚠️  Chưa có file Parquet")
        return
    
    # Nhóm files theo folder/prefix
    from collections import defaultdict
    grouped = defaultdict(list)
    
    for f in parquet_files:
        # Lấy phần đầu của đường dẫn (folder)
        parts = f.split('/')
        if len(parts) > 1:
            prefix = parts[0]
        else:
            prefix = 'root'
        grouped[prefix].append(f)
    
    # Hiển thị
    for prefix in sorted(grouped.keys()):
        print(f"📁 {prefix}/")
        for file in grouped[prefix][:10]:  # Hiển thị 10 file đầu
            print(f"   ├─ {file}")
        if len(grouped[prefix]) > 10:
            print(f"   └─ ... và {len(grouped[prefix]) - 10} file khác")


def load_topic_data(topic_name: str, limit: int = 10) -> pd.DataFrame:
    """
    Tải và hiển thị dữ liệu từ một topic cụ thể
    
    Args:
        topic_name: Tên topic ('orders', 'payments', 'shipments', 'delivery', 'users', 'products')
        limit: Số dòng hiển thị
    """
    if topic_name not in TOPICS:
        print(f"❌ Topic không tồn tại. Chọn từ: {list(TOPICS.keys())}")
        print(f"\n💡 Dùng list_all_parquet_files() để xem tất cả files")
        return None
    
    search_term = TOPICS[topic_name]
    
    # Tìm file liên quan - dùng multiple patterns
    matching_files = [f for f in parquet_files if search_term in f.lower()]
    
    if not matching_files:
        print(f"❌ Không tìm thấy file cho topic '{topic_name}'\n")
        print(f"📌 Các từ khóa tìm: '{search_term}'")
        print(f"📊 Tổng files trong parquet_files: {len(parquet_files)}")
        
        if parquet_files:
            print(f"\n💡 Các files tìm được ({len(parquet_files)} total):")
            list_all_parquet_files()
        else:
            print(f"\n⚠️  Chưa có Parquet files. Hãy chạy Spark job!")
        return None
    
    # Lấy file mới nhất (sắp xếp theo tên)
    matching_files.sort()
    file_to_load = matching_files[-1]  # Lấy file cuối (mới nhất)
    
    try:
        print(f"\n{'='*80}")
        print(f"📊 TOPIC: {topic_name.upper()}")
        print(f"{'='*80}\n")
        print(f"📁 File: {file_to_load}")
        
        # Download và load file
        response = client.get_object(bucket_name, file_to_load)
        data = response.read()
        df = pd.read_parquet(io.BytesIO(data))
        
        # Thông tin cơ bản
        print(f"\n📈 Thông tin:\n")
        print(f"   Tổng bản ghi: {len(df):,}")
        print(f"   Số cột: {df.shape[1]}")
        print(f"   Dung lượng: {len(data) / 1024 / 1024:.2f} MB")
        
        # Schema
        print(f"\n🔍 Schema:")
        for i, (col, dtype) in enumerate(df.dtypes.items(), 1):
            print(f"   {i:2}. {col:<25} ({str(dtype):<15})")
        
        # Dữ liệu mẫu
        print(f"\n📋 Dữ liệu mẫu ({limit} bản ghi đầu):")
        print(df.head(limit).to_string())
        
        # Thống kê
        if len(df.select_dtypes(include=['number']).columns) > 0:
            print(f"\n📊 Thống kê cơ bản:")
            print(df.describe().T.to_string())
        
        print(f"\n" + "="*80)
        
        return df
        
    except Exception as err:
        print(f"❌ Lỗi khi tải: {err}")
        return None

def print_all_topics(limit: int = 5):
    """
    In dữ liệu từ tất cả các topics
    
    Args:
        limit: Số dòng hiển thị cho mỗi topic
    """
    print(f"\n🚀 HIỂN THỊ TẤT CẢ TOPICS (mỗi topic hiển thị {limit} dòng)\n")
    
    topic_data = {}
    for topic_key in TOPICS.keys():
        df = load_topic_data(topic_key, limit=limit)
        if df is not None:
            topic_data[topic_key] = df
    
    return topic_data

print("\n✅ Hàm print topic sẵn sàng!")
print(f"\n📌 Topic có sẵn: {list(TOPICS.keys())}")
print("\nCách sử dụng:")
print("  - list_all_parquet_files()               # Xem tất cả files")
print("  - load_topic_data('orders', limit=10)    # In 10 dòng từ orders")
print("  - load_topic_data('payments', limit=5)   # In 5 dòng từ payments")
print("  - print_all_topics(limit=3)              # In tất cả topics")


✅ Hàm print topic sẵn sàng!

📌 Topic có sẵn: ['orders', 'payments', 'shipments', 'delivery', 'users', 'products']

Cách sử dụng:
  - list_all_parquet_files()               # Xem tất cả files
  - load_topic_data('orders', limit=10)    # In 10 dòng từ orders
  - load_topic_data('payments', limit=5)   # In 5 dòng từ payments
  - print_all_topics(limit=3)              # In tất cả topics


In [24]:
# Chọn topic + số dòng hiển thị
topic_to_view = 'products'    # Đổi topic ở đây
records_limit = 10              # Đổi số dòng ở đây

df_selected = load_topic_data(topic_to_view, limit=records_limit)


📊 TOPIC: PRODUCTS

📁 File: warehouse/bronze/raw_events/data/event_time_day=2026-04-03/event_source=demo.public.products/00066-887-3795892e-88ff-4760-b297-1d0dd803866d-2-00001.parquet

📈 Thông tin:

   Tổng bản ghi: 103
   Số cột: 7
   Dung lượng: 0.01 MB

🔍 Schema:
    1. event_source              (str            )
    2. event_time                (datetime64[us, UTC])
    3. schema_id                 (int32          )
    4. payload_size              (int32          )
    5. json_payload              (str            )
    6. partition                 (int32          )
    7. offset                    (int64          )

📋 Dữ liệu mẫu (10 bản ghi đầu):
           event_source                       event_time  schema_id  payload_size                                                                                                                                                                                                                                                                  